# RAG - Build Vector DB From Scratch

It's ok if we have document_vectors of 100 chunks. But what if 100,000,000 chunks???

Look at this Brute Force approach:

Given `query_vector` and `document_vectors`.

```python
best_score = -inf

for vector in document_vectors:
    score = cosine_similarity(query_vector, vector)

    if score > best_score:
        best_score = score
```

You have to compare query_vector to 100,000,000 documents with O(n) complexity. **HUGE**!!!

> To solve this problem, we build **indexes**.

While other queries like SQL search for exact values, Vector Search asks

> Which vectors are most similar?

At this point, we use technique **Nearest Neighbour Search** (e.g., k-NN).

There are 2 k-NN:

- Exact k-NN: optimal but slow.
- Approximate NN (ANN): something similar but faster.

> Simple ANN idea: **Clustering** > **Inverted File Index (IVF)**, with each cluster has a centroid (e.g., FAISS library by Meta, HNSW).



## Hierarchical Navigable Small World (HNSW)

Start at random node, keep moving to a closer neighbour, eventually reach the nearest region.

HNSW builds layers:

> from top to bottom: few nodes > more nodes > ... > all vectors.

Process:

1. Graph Algorithms
2. Greedy Search
3. Hierarchical Shorcuts

Complexity: O(log n)

## Init

In [1]:
import numpy as np

## Pipeline

In [5]:
A = tuple([1, 1])
B = tuple([2, 1])
C = tuple([3, 1])
D = tuple([4, 1])
E = tuple([5, 1])

# A --- B --- C --- D --- E are a straight line
# Connect each vector to its nearest neighbours

graph = {
    A: [B],
    B: [A, C],
    C: [B, D],
    D: [C, E],
    E: [D]
}

In [7]:
def distance(a: tuple, b: tuple) -> tuple:
    return np.linalg.norm(a - b)

In [8]:
def search(graph: dict, vectors: tuple, start: tuple, query: tuple) -> tuple:
    current = start

    while True:
        current_dist = distance(query, vectors[current])

        best_node = current

        for neighbour in graph[current]:
            d = distance(query, vectors[neighbour])

            if d < current_dist:
                best_node = neighbour
                current_dist = d

        if best_node == current:
            break
            
        current = best_node

    return current

To prevent Local minima problem, use **prioritised BFS** with priority queue.

### Hierarchy

Create **a map of the map**.

- Layer 2

```
    G
  /   \
A       M
```

- Layer 1

```
A B C D E F G H I J K L M
```

- Layer 0

```
All vector
```

In [9]:
import random

Generate ramdom level

In [13]:
level = 0

while random.random() < .5:
    level += 1

In [14]:
print(level)

2


With this approach, we expect:

Level 0:
1,000,000

Level 1:
500,000

Level 2:
250,000

Level 3:
125,000

Then at each level: greedy search > find closest node > descend to lower level.

In [15]:
class Node:
    def __init__(self, vector, level):
        self.vector = vector
        self.level = level
        self.neighbours = {}

Full process

1. Generate random level
2. Search existing graph
3. Find nearest nodes
4. Connect both directions
5. Repeat for every level

Next:

1. Brute-force cosine search
2. k-NN graph + greedy search
3. k-NN graph + beam search
4. Hierarchical graph (mini-HNSW)

## Brute-force Cosine Search (Exact k-NN)

Search Engine over vectors.

Check every vector > compute cosine similarity > get the highest.

In real production, we index vectors. Instead of storing:

```python
{
    vector1,
    vector2,
    vector3
}
```

We store:

```python
{
    {
        "id": 1,
        "text": "",
        "embedding": []
    },
    {
        "id": 2,
        "text": "",
        "embedding": []
    }, ...
}
```

with `embedding` is used for searching, LLM still needs to return the `text`.

We can start with this simple Data Structure.

### Data Structure

In [16]:
class DocumentChunk:
    
    def __init__(self, chunk_id, text, embedding):
        self.chunk_id = chunk_id
        self.text = text
        self.embedding = embedding

### Search Engine

Store vectors -> Search vectors

In [17]:
class BruteForceVectorIndex:

    def __init__(self):
        self.documents = []

### Adding Documents

In [ ]:
class BruteForceVectorIndex:

    def __init__(self):
        self.documents = []
    
    def add(self, chunk):
        self.documents.append(chunk)

### Searching

In [18]:
class BruteForceVectorIndex:

    def __init__(self):
        self.documents = []
    
    def add(self, chunk):
        self.documents.append(chunk)
    
    def search(self, query_embedding):
        best_score = float("-inf")
        best_doc = None

        for doc in self.documents:
            score = cosine_similarity(query_embedding, document.embedding)

            if score > best_score:
                best_score = score
                best_doc = doc
        
        return best_doc

We can also get the Top-k of the docs.

## k-NN Graph + Greedy Search

If you're asked what is the nearest cafe, you don't walk to every cafe, measure distance and get the nearest one. That's dumb!

We human don't do it. Instead, we elevate one property of geography: *nearby place tend to connect to nearby place*.

To build a graph, we use k-NN to connect k neighbours to each of the node, which costs $O(n^2)$ but it's ok since it's only done once in indexing step.

### Build Graph

Algorithm

Double looping
- Time: $O(n^2)$

```
GET_NEIGHBOURS(graph)
    for node in graph
        similarities = []

        for other_node in graph
            if other_node == node
                continue
            
            score = cosine_similarity(node, other_node)
            similarities.append((score, other_node))
        
        SORT(similarities)

        keep the top k neighbours
```

In [ ]:
def cosine_similarity:
    return None

class GraphNode:

    def __init__(self, chunk_id, text, embedding):
        self.chunk_id = chunk_id
        self.text = text
        self.embedding = embedding
        self.neighbours = []
    
    def get_neighbours(self, graph: list[GraphNode], k: int):
        for node in graph:
            similarities = []

            for other_node in graph:
                if other_node == node:
                    continue

                score = cosine_similarity(node.embedding, other_node.embedding)
                similarities.append((score, other_node))
            
            sorted_similarities = sorted(similarities, key=lambda x: x[0])
            
            self.neighbours = [x[1]] for x in sorted_similarities[-k:]
            

### Searching

One problem with Greedy search is *Local Minima*.

## k-NN Graph + Beam Search

Like Djikstra's with priority queue but each layer in the queue, only keep best k number of nodes.

Algorithm

Beam Search
- Time: [TBD]
- Space: O(n)

```
BEAM_SEARCH(query_embedding, root)
    queue = [] // piority queue
    queue.push(root)
    visited = []

    while queue not empty
        best = queue.pop

        for neighbour in best.neighbours
            if neighbour not in visited
                score = COSINE_SIMILARITY(query_embedding, neighbour)
                queue.push((neighbour, score))

                PRUNE(queue)
        
    return best node
```


What makes Beam Search still not sufficient???

The graph itself is inefficient. -> It needs to be easier to navigate.

## Hierarchical Graph (mini-HNSW)

In real life, when you want to travel from A to B, you go from:

> Motorway > Major Road > Local Road > Street

- How to add shortcuts to graph?
- Hub and spoke, airports
- Layers
  - Layer 0: all vectors - 50%
  - Layer 1: fewer nodes - 25%
  - Layer 2: fewer nodes - 12.5%
  - ...
- At each level, get best node, then go down one level.

In [3]:
from src import HNSWNode, HNSWIndex

Questions remained:

> How expensive is insertion?


The **HNSWIndex > insert function** does:
1. Search entire graph
2. Find neighbours
3. Connect



## Problems With Current Approach

### Problem 1 - Beam Search

How large should the beam (in beam search) be?

Say at a layer, we have looked at `beam_width` of nodes, then keep going down from them. Then we could probably not able to search for what we actually need.

```
                #
            /
    # # # #                 #
    -------
    beam_width
        /
# # #  # # #
```

So Beam Search has a weakness that to explore *locally*.

### Problem 2 - Performance Drop

Now we try to increase `beam_width` to cover more nodes. Accuracy increases, but much more memory and computation.

### Problem 3 - Graph is Bad

Worst case: linear graph A - B - C - D - E.

What we want

```
    A
   /|\
  B C D
/   |   \
.......
```

This is the idea of *Small World* phenomenom, we add flights between countries, instead of letting people walk from one to another.

### Problem 4 - Navigable Small World NSW?

To solve problem 3, instead of connect all nodes together, we only connect nearest neighbours to each.

But there's still a problem:

> Even if we have NSW, hopping between 100,000 vectors is insane.

It's like going from street to street, not leveraging motorway, major road, local road.

## HNSW's Central Idea

Instead of one graph, we build many in layers

- Top layer: Very sparse
- Layer 0: every point

### Engineering Improvements

- M
- efConstruction
- efSearch
- neighbour heuristic
- pruning

### Exponentiallity Decreasing

Suppose each layer keeps 50% of nodes:

> 1,000,000 -> 500,000 -> 250,000 -> 125,000...

It's still enormous. Now we keep only 10%:

> 1,000,000 -> 100,000 -> 10,000 -> 1,000

Free search. Probability of appearing in higher layers.